<!-- FILE MAP | Colab wrapper for the canonical runner. Thin on purpose: all training logic
     lives in train.py + src/training/. This notebook only (1) mounts Drive, (2) pulls code
     and fetches data/checkpoints, (3) runs train.py. [CHANGE ME] pick the model/seed by
     editing configs/run.yaml, NOT here. -->

# Train (Colab wrapper)

One runner, one protocol — every model is trained and measured the same way.

**To choose the model/seed/epochs, edit `configs/run.yaml`** (`run.model` = `unet` | `sam_lora` | `medsam`),
commit + push, then run the three cells below. Results (metrics.json, mask overlays, run.log)
are written to `results/<model>/seed<seed>/` and mirrored to your Drive results folder.

In [ ]:
# 1. Mount Google Drive (results + checkpoints are mirrored here).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 2. Pull latest code + set up environment (deps, dataset, model checkpoint).
import os, sys, hashlib, zipfile, shutil
from pathlib import Path

REPO = '/content/msu2026summer_final_project'
if not os.path.exists(REPO):
    !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
%cd {REPO}
!git fetch --quiet origin && git reset --hard origin/main
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# --- read which model the run is configured for ---
from src.config import load_run_config, build_run_plan
cfg = load_run_config('configs/run.yaml')
plan = build_run_plan(cfg)
print(f'Configured run: model={plan.model}  backbone={plan.backbone}  seed={plan.seed}')

# --- datasets (PraNet train + test packages) ---
DATA_ROOT = Path('data/polyp'); DATA_ROOT.mkdir(parents=True, exist_ok=True)
def _dl_data(file_id, zip_name):
    import gdown
    target = DATA_ROOT / zip_name.replace('.zip', '')
    if target.exists():
        print(f'{target.name}: already present'); return
    zp = f'/tmp/{zip_name}'
    gdown.download(id=file_id, output=zp, quiet=False, fuzzy=True)
    tmp = Path(f'/tmp/x_{target.name}'); tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as zf: zf.extractall(tmp)
    ex = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(ex) == 1 and ex[0].is_dir(): shutil.move(str(ex[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for p in ex: shutil.move(str(p), str(target / p.name))
    print(f'Done -> {target}')
_dl_data('1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb', 'TrainDataset.zip')
_dl_data('1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao', 'TestDataset.zip')

# --- model checkpoint (only the backbone the config needs) ---
def _md5(path, chunk=8*1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

if plan.model == 'sam_lora':
    fname = cfg['sam']['checkpoint']
    urls = {'sam_vit_h_4b8939.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
            'sam_vit_b_01ec64.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'}
    if not os.path.exists(fname):
        print(f'Downloading SAM checkpoint {fname}...')
        !wget -q --show-progress {urls[fname]} -O {fname}
    print(f'SAM checkpoint ready: {fname}')
elif plan.model == 'medsam':
    fname = cfg['medsam']['checkpoint']
    MEDSAM_URL = 'https://zenodo.org/records/10689643/files/medsam_vit_b.pth'
    MEDSAM_MD5 = '3bb6db55bd0c9ca30b61248bca72f8d6'
    if os.path.exists(fname) and (os.path.getsize(fname) < 300_000_000 or _md5(fname) != MEDSAM_MD5):
        os.remove(fname)
    if not os.path.exists(fname):
        print('Downloading MedSAM ViT-B checkpoint from Zenodo (~375 MB)...')
        !wget -q --show-progress {MEDSAM_URL} -O {fname}
    assert _md5(fname) == MEDSAM_MD5, 'MedSAM MD5 mismatch'
    print(f'MedSAM checkpoint ready: {fname}')
else:
    print('U-Net: encoder weights download automatically via segmentation-models-pytorch.')

In [ ]:
# 3. Train. All logic is in train.py; edit configs/run.yaml to change what runs.
!python train.py --config configs/run.yaml